In [ ]:
import numpy as np
import pandas as pd
from demand_platform.data.ingestion import load_dataset
from demand_platform.data.schema import confirm_mapping, interpret_schema
from demand_platform.data.standardization import standardize

NAN = np.nan
PATH = "../datas/processed/Week 0 - 2024-04-08 - Sales.csv"

with open(PATH, "rb") as f:
    raw = load_dataset(f, PATH)
vn2 = standardize(raw, confirm_mapping(interpret_schema(raw), "Store", "Product"))
vn2

In [ ]:
def make(series):
    return pd.concat([
        pd.DataFrame({"store_id": s, "product_id": p,
                      "date": pd.date_range("2021-04-12", periods=len(v), freq="W-MON"),
                      "sales": pd.Series(v, dtype="float64")})
        for (s, p), v in series.items()
    ], ignore_index=True)


# toy1: data problems VN2 doesn't have
toy1 = make({
    ("1", "A"): [0, 0, 2, 0, 0, 3, NAN, 0],  # late start, NaN, trailing zero
    ("1", "B"): [0, 0, 0, 0],                # never sold
    ("2", "A"): [0, -1, 4, 0, 5],            # a return before the first sale
})
toy1

In [ ]:
# toy2: shapes that break statistics
toy2 = make({
    ("1", "A"): [3, 3, 3, 3, 3],             # constant: std 0
    ("1", "B"): [7],                         # single row
    ("1", "C"): [1.0] * 20 + [100.0, 1.0],   # one spike
    ("2", "A"): [5, 0, 0, 0, 0, 0, 0, 5],    # long zero run in the middle
})
toy2